# Fast ViT on CIFAR-10 — Demo

Train a small **Vision Transformer** on CIFAR-10 quickly by **distilling from a frozen pretrained CNN teacher**. We compare four losses, all using the **same** student and data pipeline:

| Method | Loss | Saved JSON tag |
|---|---|---|
| Baseline | CE only | `baseline` |
| Vanilla KD | CE + KL(student ‖ teacher) | `vanilla_kd` |
| DKD | CE + α·TCKD + β·NCKD | `dkd` |
| DKD + Feature | DKD + MSE(proj(student CLS), teacher penult.) | `feat_dkd` |

The metric is **wall-clock time to 85% test accuracy** (lower is better).

**What this demo does:**
1. Loads the teacher and student, prints sizes.
2. Defines all four loss functions in one cell so you can read them side by side.
3. Runs **5 epochs** of the strongest method (DKD + Feature) live to prove the pipeline works.

For full benchmark runs and the headline comparison plot see the reference notebooks listed at the bottom. For reproduction details see [`README.md`](README.md). For the open-benchmark competition ruleset see [`BENCHMARK.md`](BENCHMARK.md).

## 1. Setup

In [ ]:
import sys, math, time
from pathlib import Path

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import torch
import torch.nn as nn
import torch.nn.functional as F

from src.model import count_parameters
from src.utils import get_device, get_cifar10_loaders, get_model, set_seed, validate

torch.set_float32_matmul_precision("high")
device = get_device()
print("torch", torch.__version__, "device", device)

## 2. Data, teacher, student

Teacher = pretrained CIFAR-10 ResNet-56 via `torch.hub` (frozen). Student = the small ViT defined in `src/model.py`.

In [ ]:
DEMO_BATCH = 256
NUM_WORKERS = 4

train_loader, test_loader = get_cifar10_loaders(
    data_dir=str(PROJECT_ROOT / "data"),
    batch_size=DEMO_BATCH,
    num_workers=NUM_WORKERS,
    augment_train=True,
    use_randaugment=True,
    persistent_workers=NUM_WORKERS > 0,
)

teacher = torch.hub.load("chenyaofo/pytorch-cifar-models", "cifar10_resnet56", pretrained=True)
teacher.eval()
for p in teacher.parameters():
    p.requires_grad_(False)
teacher = teacher.to(device)

_, teacher_acc = validate(teacher, test_loader, nn.CrossEntropyLoss(), device)

def make_student():
    return get_model(patch_size=4, embed_dim=192, depth=6, num_heads=6, dropout=0.0)

_student_for_count = make_student()
print(f"teacher (ResNet-56) test acc : {teacher_acc:.2f}%")
print(f"student (ViT) params         : {count_parameters(_student_for_count):,}")

## 3. The four losses, side by side

All four are pluggable into the same training loop — only this function changes.

- **CE** sees only the true label.
- **Vanilla KD** also sees the teacher's full softmax (`KL(student ‖ teacher)` at temperature `T`).
- **DKD** decouples that into the **target-class** signal (TCKD = 2-bin KL on `[P(gt), P(rest)]`) and the **non-target dark-knowledge** signal (NCKD = KL over the wrong classes only).
- **DKD + Feature** also matches the teacher's **penultimate feature vector** with the student's CLS token through a small learnable projection — this is the strongest method in our experiments.

In [ ]:
def loss_ce(zs, y):
    return F.cross_entropy(zs, y, label_smoothing=0.1)


def loss_vanilla_kd(zs, zt, y, T=4.0, kd_weight=4.0):
    ce = F.cross_entropy(zs, y)
    kd = F.kl_div(
        F.log_softmax(zs / T, dim=1),
        F.softmax(zt / T, dim=1),
        reduction="batchmean",
    ) * (T * T)
    return ce + kd_weight * kd


def _gt_mask(z, y):
    return torch.zeros_like(z).scatter_(1, y.view(-1, 1), 1).bool()


def loss_dkd(zs, zt, y, alpha=1.0, beta=8.0, T=4.0):
    gt = _gt_mask(zs, y)
    other = ~gt
    ps = F.softmax(zs / T, dim=1)
    pt = F.softmax(zt / T, dim=1)
    ps2 = torch.cat([(ps * gt).sum(1, keepdim=True), (ps * other).sum(1, keepdim=True)], 1)
    pt2 = torch.cat([(pt * gt).sum(1, keepdim=True), (pt * other).sum(1, keepdim=True)], 1)
    tckd = F.kl_div(ps2.clamp_min(1e-12).log(), pt2, reduction="sum") * T * T / y.shape[0]
    pt_nt = F.softmax(zt / T - 1000.0 * gt.float(), dim=1)
    ps_nt = F.log_softmax(zs / T - 1000.0 * gt.float(), dim=1)
    nckd = F.kl_div(ps_nt, pt_nt, reduction="sum") * T * T / y.shape[0]
    ce = F.cross_entropy(zs, y)
    return ce + alpha * tckd + beta * nckd


def loss_dkd_plus_feature(zs, zt, y, s_feat, t_feat, proj, feat_weight=4.0):
    base = loss_dkd(zs, zt, y)
    feat = F.mse_loss(proj(s_feat), t_feat.flatten(1).detach())
    return base + feat_weight * feat


print("loss functions defined: loss_ce, loss_vanilla_kd, loss_dkd, loss_dkd_plus_feature")

## 4. Live mini-training (5 epochs, DKD + Feature)

Trains the **strongest** method end-to-end for 5 epochs to prove the pipeline works. This is **not** a benchmark run; the headline numbers come from §5. Expect roughly **1–3 minutes** on a recent CUDA GPU.

In [ ]:
DEMO_EPOCHS = 5
LR = 1e-3

set_seed(42, deterministic=False)
student = make_student().to(device)

# --- Feature grabbers (no model surgery; pure hooks) -------------------------
class _Grab:
    def __init__(self):
        self.feat = None
        self._h = None
    def post(self, m):
        self._h = m.register_forward_hook(lambda _m, _i, out: setattr(self, "feat", out))
        return self
    def pre(self, m):
        self._h = m.register_forward_pre_hook(lambda _m, inp: setattr(self, "feat", inp[0]))
        return self
    def remove(self):
        if self._h is not None:
            self._h.remove(); self._h = None

# Student feature = input to head (CLS token after the final LayerNorm).
s_grab = _Grab().pre(student.head)

# Teacher feature = output of penultimate global-pool layer.
for name in ("avgpool", "global_pool", "gap"):
    if hasattr(teacher, name):
        t_grab = _Grab().post(getattr(teacher, name))
        break
else:
    t_grab = _Grab().pre(teacher.fc)

# Probe teacher feature dim with a 2-sample forward.
with torch.no_grad():
    xb, _ = next(iter(train_loader))
    teacher(xb.to(device)[:2])
T_DIM = t_grab.feat.flatten(1).shape[1]
S_DIM = student.head.in_features

proj = nn.Linear(S_DIM, T_DIM, bias=False).to(device)
nn.init.trunc_normal_(proj.weight, std=0.02)

opt = torch.optim.AdamW(
    list(student.parameters()) + list(proj.parameters()),
    lr=LR,
    weight_decay=0.01,
)

print(f"student feat dim {S_DIM} → teacher feat dim {T_DIM}, projection ready")

# --- Training loop -----------------------------------------------------------
t0 = time.perf_counter()
for ep in range(1, DEMO_EPOCHS + 1):
    student.train()
    running = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad(set_to_none=True)
        zs = student(x)
        with torch.no_grad():
            zt = teacher(x)
        loss = loss_dkd_plus_feature(zs, zt, y, s_grab.feat, t_grab.feat, proj)
        loss.backward()
        nn.utils.clip_grad_norm_(
            list(student.parameters()) + list(proj.parameters()), 1.0
        )
        opt.step()
        running += loss.item()
    _, acc = validate(student, test_loader, nn.CrossEntropyLoss(), device)
    elapsed = time.perf_counter() - t0
    print(
        f"epoch {ep}/{DEMO_EPOCHS} | loss {running / len(train_loader):.3f} "
        f"| test {acc:.2f}% | elapsed {elapsed:.1f}s"
    )

s_grab.remove(); t_grab.remove()
print("\npipeline is working end-to-end — for a full benchmark run see notebooks/feature_distillation.ipynb")

## 5. Summary

- The **same** ViT student trained from scratch is **slower** to reach 85% test accuracy than the same student distilled from a CNN teacher.
- **DKD** beats vanilla KD by separating target-class behavior from non-target dark knowledge.
- **DKD + feature matching** is the fastest method we measured, because feature alignment gives a denser per-sample regression signal than the 10-way softmax.

**More:**
- Reproduction instructions: [`README.md`](README.md)
- Open-benchmark ruleset (two tracks: no-distillation and CNN-teacher distillation): [`BENCHMARK.md`](BENCHMARK.md)
- Reference notebooks for the four runs above:
  - `notebooks/no_distillation_baseline.ipynb`
  - `notebooks/kd_teacher_comparison.ipynb`
  - `notebooks/dkd_distillation.ipynb`
  - `notebooks/feature_distillation.ipynb`